# RTC-SDD Full Train/Dev Pretrained Backbone Benchmark
Frozen encoder + fixed linear probe. Full RTCFake: 75,785 Train + 15,204 Dev. No Progress/Final use. Use one T4. **Do not run the old 04 screening notebook for this experiment.**

Upload this new script/Notebook to the `research/pretrained-backbone-benchmark` branch before running. Clone branch with existing Kaggle Secret `GITHUB_TOKEN`. Do not print secrets.

In [ ]:
from pathlib import Path
import os, sys, subprocess, base64, torch
from kaggle_secrets import UserSecretsClient
ROOT = Path('/kaggle/working/RTC-SDD-Research')
DATA_ROOT = Path('/kaggle/input/datasets/a7medlabib/rtcfake-packed-private')
BRANCH = 'research/pretrained-backbone-benchmark'
assert torch.cuda.is_available()
assert (DATA_ROOT/'train_label.txt').is_file() and (DATA_ROOT/'dev_label.txt').is_file()
if not (ROOT/'.git').exists():
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    assert token
    env = os.environ.copy()
    env['GIT_CONFIG_COUNT']='1'
    env['GIT_CONFIG_KEY_0']='http.extraheader'
    env['GIT_CONFIG_VALUE_0']='AUTHORIZATION: basic ' + base64.b64encode(('x-access-token:'+token).encode()).decode()
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch','https://github.com/AhmedLabib1/RTC-SDD-Research.git',str(ROOT)],check=True,env=env)
assert subprocess.check_output(['git','-C',str(ROOT),'branch','--show-current'], text=True).strip()==BRANCH
assert (ROOT/'scripts/benchmark_pretrained_backbones_full.py').is_file(), 'Push the new full benchmark script to this branch first'
print('Ready:',ROOT,'GPU:',torch.cuda.get_device_name(0),'Python:',sys.version.split()[0])

The repository's editable pip install requires Python 3.10; Kaggle 3.13 uses `PYTHONPATH` instead. Uses existing torch, sklearn, transformers and soundfile. Verify the checkpoint cards for competition eligibility before trusting any ranking.

In [ ]:
import importlib
for name in ('torch','transformers','sklearn','soundfile','huggingface_hub'):
    importlib.import_module(name)
    print('PASS',name)
env=os.environ.copy()
env['PYTHONPATH']=str(ROOT/'src')
env['CUDA_VISIBLE_DEVICES']='0'
subprocess.run([sys.executable,'-m','rtcsdd','preflight','--dataset-root',str(DATA_ROOT),'--require-audit-counts'],check=True,env=env)
print('Dataset preflight PASS')

## Full evaluation on ONE T4
This can take substantial GPU time. Each model writes an atomic chunk after 256 files; rerunning resumes cached chunks **if files persist**. Kaggle `/kaggle/working` is session-local; save a Notebook version/output or move `full_benchmark` into your own private Kaggle dataset between sessions for cross-session resume. Do not publish extracted embeddings or raw data. One model failure does not erase earlier files.

In [ ]:
import subprocess, sys, json, os
from pathlib import Path
MODELS=[
 'facebook/wav2vec2-base',
 'facebook/hubert-base-ls960',
 'microsoft/wavlm-base-plus',
 'facebook/data2vec-audio-base',
 'facebook/wav2vec2-xls-r-300m',
]
OUT=Path('/kaggle/working/full_benchmark')
OUT.mkdir(parents=True,exist_ok=True)
errors={}
for name in MODELS:
    print('\n===',name,'===',flush=True)
    cmd=[sys.executable,str(ROOT/'scripts/benchmark_pretrained_backbones_full.py'),
         '--data-root',str(DATA_ROOT),'--model',name,
         '--train-samples','0','--dev-samples','0',
         '--batch-size','2','--chunk-size','256','--workers','2',
         '--seed','1234','--output-dir',str(OUT)]
    p=subprocess.run(cmd,env=env)
    if p.returncode:
        errors[name]=p.returncode
        print('FAILED:',name,'code=',p.returncode,flush=True)
print('Failed models:',errors)

In [ ]:
import json
import pandas as pd
rows=[]
reference=None
for name in MODELS:
    path=OUT/name.replace('/','__')/'result.json'
    if not path.exists():
        print('INCOMPLETE',name)
        continue
    r=json.loads(path.read_text())
    current=(r['train_manifest_sha256'],r['dev_manifest_sha256'],r['train_count'],r['dev_count'])
    if reference is None: reference=current
    assert current==reference, 'Dataset manifest mismatch: '+name
    assert r['train_count']==75785 and r['dev_count']==15204
    m=r['metrics']
    rows.append({'model':name,'checkpoint_commit':r['checkpoint_commit'],
      'overall_f1':m['all_dev']['macro_f1'],'online_f1':m['view/online']['macro_f1'],
      'offline_f1':m['view/offline']['macro_f1'],
      'spoof_recall':m['all_dev']['spoof']['recall'],
      'bonafide_recall':m['all_dev']['bonafide']['recall'],
      'run_wall_s':r['seconds']['run_wall']})
if rows:
    df=pd.DataFrame(rows).sort_values('online_f1',ascending=False)
    display(df)
    df.to_csv(OUT/'full_benchmark_summary.csv',index=False)
else:
    print('No models completed')

**Reporting**: This is a frozen-feature probe benchmark, not fine-tuned AASIST or Codabench, and not equivalent to Clean/Noisy official scores. Do not select a final backbone only from probe ranking. Preserve the JSON results and checkpoint provenance. Do not commit `.npz` features to GitHub.